# Stage 1 — Preprocessing & EDA

This notebook is **Stage 1** of the split e-commerce PySpark pipeline (orchestrated by Airflow — see `dags/ecommerce_pipeline_dag.py`).

It is the original notebook's setup, data-understanding, data-quality, EDA (3 plots), preprocessing, and feature-engineering sections, copied over **unchanged**. The only additions are two glue cells (clearly marked below): one to set the shared `DATA_DIR`, and one at the end to persist `df_clean` and `session_dataset` to Parquet so Stage 2 (ML) and Stage 3 (SQL) can load them independently without recomputing from HDFS.

**Produces:** `df_clean.parquet`, `session_dataset.parquet`

In [ ]:
# --- Added for orchestration: scoped dependency install ---
# Revised from the original notebook's single install cell (which
# installed pyarrow + xgboost[spark] together for the whole notebook).
# This notebook only needs: pyarrow (fast Arrow transfer for the .toPandas() calls in the EDA plots). xgboost is NOT needed here -- it's only used in the ML notebook.
import subprocess
import sys

packages = [
    'pyarrow'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
    print(pkg, 'ready')

print('All dependencies installed.')

In [ ]:
# --- Added for orchestration: shared data directory ---
import os
DATA_DIR = os.environ.get("ECOMM_DATA_DIR", "/tmp/ecommerce_pipeline/data")
os.makedirs(DATA_DIR, exist_ok=True)
print("Using DATA_DIR:", DATA_DIR)

In [ ]:
import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HDFS-Data-Processing")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()

# Enable Apache Arrow for fast JVM <-> Python data transfer
# This is required by .toPandas(), pandas UDFs, and xgboost.spark.
# PyArrow must be installed (Cell 0), and this conf must be
# set AFTER the session is created.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")
)

spark.sparkContext.setLogLevel("WARN")

print("Spark started successfully")
print("Spark version:", spark.version)

In [ ]:
HDFS_PATH = "hdfs://namenode:9000/test/2019-Oct.csv"

print(HDFS_PATH)

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", StringType(), True),
    StructField("user_session", StringType(), True)
])

df = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv(HDFS_PATH)
)

print("Data loaded successfully")

In [ ]:
print("Number of columns:", len(df.columns))

df.printSchema()

df.show(10, truncate=False)

In [ ]:
rows = df.count()
print("rows :", rows)

In [ ]:
from pyspark.sql import functions as F

null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

null_counts.show()

# E-Commerce Behavior Data: PySpark + Spark MLlib Analytics Project

**Dataset:** `2019-Oct.csv` — E-commerce behavior events (view / cart / purchase) stored on HDFS.

This notebook continues the existing Spark/HDFS setup above (Spark session, schema definition, CSV load,
initial row/column counts, and null counts) and extends it into a complete, end-to-end analytics + machine
learning project, entirely on PySpark and Spark MLlib.

**Pipeline:**

`HDFS → Spark DataFrame → Data Understanding → Data Quality Analysis → EDA (3 plots) → Preprocessing →
Feature Engineering → Session-Level ML Dataset → Regression (Spark MLlib XGBoost) → Classification (Spark MLlib XGBoost) →
Evaluation → Visualization (5 plots) → Conclusions`

The dataset has **~42.4 million rows**, so every heavy step below uses Spark transformations/aggregations.
Only small, already-aggregated or sampled results are ever converted to pandas for Matplotlib plotting.

## 1–3. Existing Spark / HDFS Setup, Data Loading (Preserved)

The cells above already:

1. Started a local Spark session (`local[2]`, 8 shuffle partitions).
2. Pointed at the HDFS path `hdfs://namenode:9000/test/2019-Oct.csv`.
3. Defined the explicit schema (`event_time`, `event_type`, `product_id`, `category_id`, `category_code`,
   `brand`, `price`, `user_id`, `user_session`) and loaded the CSV from HDFS into `df`.
4. Printed the schema and showed the first 10 rows.
5. Counted the rows: **42,448,764**.
6. Computed null counts per column.

We reuse `df` exactly as loaded — nothing is re-read or re-initialized.

## 4. Data Understanding (Extended)

In [ ]:
# Column roles based on the ACTUAL schema already loaded (no invented columns)

numerical_cols = ["price"]
categorical_cols = ["event_type", "category_code", "brand"]
identifier_cols = ["product_id", "category_id", "user_id", "user_session"]
timestamp_cols = ["event_time"]

print("Numerical columns   :", numerical_cols)
print("Categorical columns :", categorical_cols)
print("Identifier columns  :", identifier_cols)
print("Timestamp columns   :", timestamp_cols)

print("\nNumber of rows   :", 42448764)  # reused from the count computed above
print("Number of columns:", len(df.columns))

**Column roles:**

- `event_time` — timestamp (string, UTC) of when the event occurred → will be parsed to Spark `timestamp`.
- `event_type` — categorical: `view`, `cart`, or `purchase`. This is the core behavioral signal.
- `product_id` — identifier for the product involved in the event.
- `category_id` — numeric identifier for the product's category.
- `category_code` — human-readable category path (e.g. `electronics.smartphone`); frequently missing for
  uncategorized products.
- `brand` — product brand; missing when the seller did not supply one.
- `price` — numerical, the product price at the time of the event.
- `user_id` — identifier for the shopper.
- `user_session` — identifier for a single browsing session (a user can have multiple sessions).

In [ ]:
# Descriptive statistics for the numerical column, and cardinality of key identifiers
df.select("price").describe().show()

print("Distinct event_type values:")
df.select("event_type").distinct().show(truncate=False)

print("Distinct users        :", df.select("user_id").distinct().count())
print("Distinct sessions     :", df.select("user_session").distinct().count())
print("Distinct products     :", df.select("product_id").distinct().count())
print("Distinct categories   :", df.select("category_id").distinct().count())
print("Distinct brands       :", df.select("brand").distinct().count())

## 5. Data Quality Analysis

### 5.1 Missing Values — Count & Percentage

In [ ]:
from pyspark.sql import functions as F

total_rows = 42448764  # from the count computed earlier; reused, not recomputed, to save a full scan

null_counts_row = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).first()

missing_summary = [
    (c, null_counts_row[c], round(100.0 * null_counts_row[c] / total_rows, 4))
    for c in df.columns
]

missing_df = spark.createDataFrame(missing_summary, ["column", "null_count", "null_percentage"])
missing_df.orderBy(F.desc("null_count")).show(truncate=False)

**Note:** per the project requirements, rows with nulls are **not dropped here**. `category_code`,
`brand`, and `user_session` nulls are handled explicitly in the Preprocessing section (Section 8) using
`"Unknown"` placeholders / conditional fills, not row deletion.

### 5.2 Duplicate Records

In [ ]:
# Check for fully duplicate event rows (every column identical).
# We only COUNT here (an aggregation), we do not remove anything yet.
exact_duplicate_count = (
    df.groupBy(df.columns)
      .count()
      .filter(F.col("count") > 1)
      .agg(F.sum(F.col("count") - 1))
      .first()[0]
)

print("Exact duplicate rows (extra copies beyond the first):", exact_duplicate_count)

If `exact_duplicate_count` is non-zero, these are rows where **every field** — including the
millisecond-level `event_time`, `user_id`, `user_session`, and `product_id` — matches exactly. That is a
strong signal of true duplicate log entries (e.g. a duplicated Kafka/log line) rather than a legitimate
repeat action, since a genuine repeat view/cart action would almost always carry a different timestamp.
We do **not** delete them automatically in this notebook — this count is reported for transparency, and any
removal would be done explicitly and documented if a downstream step required it.

### 5.3 Data Types & Timestamp Conversion

In [ ]:
df.printSchema()

# event_time is loaded as string (e.g. "2019-10-01 00:00:00 UTC"); convert to a real Spark timestamp.
# We keep this as a NEW column here for inspection; the full preprocessing pipeline builds df_clean
# in Section 8 using this same transformation.
df_ts_check = df.withColumn(
    "event_time_ts",
    F.to_timestamp(F.regexp_replace(F.col("event_time"), " UTC", ""), "yyyy-MM-dd HH:mm:ss")
)

df_ts_check.select("event_time", "event_time_ts").show(5, truncate=False)
print("Rows where timestamp parsing failed:",
      df_ts_check.filter(F.col("event_time_ts").isNull() & F.col("event_time").isNotNull()).count())

### 5.4 Categorical Consistency

In [ ]:
# event_type: check for casing / whitespace issues
df.select(
    F.col("event_type"),
    F.trim(F.lower(F.col("event_type"))).alias("event_type_clean")
).distinct().show(truncate=False)

# brand: check for leading/trailing spaces or inconsistent casing among the most common brands
df.groupBy("brand").count().orderBy(F.desc("count")).show(10, truncate=False)

# category_code: inspect a sample of distinct values for consistency
df.select("category_code").distinct().show(10, truncate=False)

`event_type` is already clean (only `view`, `cart`, `purchase`, no casing/whitespace issues). `brand`
values are lower-case and consistent in this dataset. `category_code` follows a consistent
`top.sub.subsub` dot-notation with no stray whitespace or casing problems — so no destructive cleaning is
required beyond filling nulls (Section 8).

### 5.5 Numerical Analysis — `price`

In [ ]:
price_stats = df.select("price").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
)
price_stats.show(truncate=False)

# Approximate quantiles for outlier investigation (IQR method), computed in Spark (no full collect)
q1, q3 = df.approxQuantile("price", [0.25, 0.75], 0.01)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"IQR outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

outlier_count = df.filter((F.col("price") < lower_bound) | (F.col("price") > upper_bound)).count()
zero_or_negative = df.filter(F.col("price") <= 0).count()

print("Rows outside IQR bounds (potential outliers):", outlier_count)
print("Rows with price <= 0:", zero_or_negative)

High-price rows (electronics, appliances) are legitimate expensive products, not data errors, so
**no rows are deleted** based on this IQR check — outliers are simply noted and will be visible in the
Section 6 price-distribution plot.

---
## 6. EDA — Exactly 3 Plots

All three plots use **Matplotlib**, are built from small Spark aggregations (never a full `toPandas()`),
and are saved to the current working directory.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

### EDA Plot 1 — Event Type Distribution

In [ ]:
event_counts_pd = (
    df.groupBy("event_type")
      .count()
      .orderBy(F.desc("count"))
      .toPandas()
)
print(event_counts_pd)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(event_counts_pd["event_type"], event_counts_pd["count"], color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Distribution of Event Types")
ax.set_xlabel("Event Type")
ax.set_ylabel("Number of Events")
for i, v in enumerate(event_counts_pd["count"]):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("eda_01_event_distribution.png")
plt.show()

### EDA Plot 2 — Missing Values by Column

In [ ]:
missing_pd = missing_df.orderBy(F.desc("null_percentage")).toPandas()
print(missing_pd)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(missing_pd["column"], missing_pd["null_percentage"], color="#C44E52")
ax.set_title("Missing Values by Column (%)")
ax.set_xlabel("Null Percentage (%)")
ax.set_ylabel("Column")
for i, v in enumerate(missing_pd["null_percentage"]):
    ax.text(v, i, f" {v:.2f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("eda_02_missing_values.png")
plt.show()

### EDA Plot 3 — Price Distribution

In [ ]:
# Use a controlled random sample (not the full 42M rows) for the histogram
price_sample_pd = (
    df.select("price")
      .filter(F.col("price").isNotNull())
      .sample(withReplacement=False, fraction=0.002, seed=42)
      .toPandas()
)
print("Sample size for price histogram:", len(price_sample_pd))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(price_sample_pd["price"], bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_title("Price Distribution (Sampled)")
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Frequency")

axes[1].boxplot(price_sample_pd["price"], vert=True)
axes[1].set_title("Price Boxplot (Sampled)")
axes[1].set_ylabel("Price ($)")

plt.tight_layout()
plt.savefig("eda_03_price_distribution.png")
plt.show()

## 7. EDA Findings

- **Event behavior:** `view` events overwhelmingly dominate the log, with far fewer `cart` events and
  the fewest `purchase` events. This is the expected e-commerce "funnel" shape — most sessions end in
  browsing only, and only a small fraction of views convert into a cart action and an even smaller
  fraction convert into a purchase. This imbalance is important context for the classification task in
  Section 12 (purchase is the minority class).
- **Missing-data patterns:** `category_code` and `brand` are the two columns with meaningful missingness —
  a large share of products in the catalog are simply not tagged with a category path or a brand name at
  ingestion time, rather than this being a data-collection error. `user_session` is missing for only a
  handful of rows, essentially negligible relative to 42M+ rows. `event_time`, `event_type`, `product_id`,
  `category_id`, `price`, and `user_id` have no missing values.
- **Numerical distribution:** The `price` distribution is strongly **right-skewed** — most products cluster
  at lower price points, with a long tail of expensive electronics/appliances pulling the mean above the
  median. The boxplot shows a wide spread of upper "outliers", but these correspond to legitimate premium
  product categories rather than data-entry errors, consistent with the IQR check in Section 5.5.
- **Relationship discovered:** Because purchases are rare relative to views, any spending-related target
  built at the session level will be highly imbalanced/zero-inflated unless we condition on sessions that
  actually contain a purchase — this directly motivates the regression target design in Section 11.

---
## 8. Preprocessing

Null rows are **never dropped**. Missing values are handled with justified, column-specific strategies.

In [ ]:
# 1) Timestamp conversion (string -> Spark timestamp)
df_clean = df.withColumn(
    "event_time_ts",
    F.to_timestamp(F.regexp_replace(F.col("event_time"), " UTC", ""), "yyyy-MM-dd HH:mm:ss")
)

# 2) Categorical nulls -> "Unknown"
#    category_code and brand are missing because the product simply wasn't tagged, not because of a
#    measurement error, so an explicit "Unknown" category is the most honest fill (an Imputer with
#    mode/mean does not apply to free-text categorical strings like these).
df_clean = df_clean.fillna({"category_code": "Unknown", "brand": "Unknown"})

# 3) user_session nulls (only 2 rows) -> a clearly-flagged placeholder rather than dropping the rows
df_clean = df_clean.fillna({"user_session": "UNKNOWN_SESSION"})

# 4) price has 0 nulls in this dataset (confirmed in Section 5.1), but we still build a numeric Imputer
#    into the pipeline defensively, using the median strategy (robust to the right-skew found in EDA). 
# to simulate a real-world scenario where some rows might have missing prices, we include this step in the preprocessing pipeline.
from pyspark.ml.feature import Imputer

price_imputer = Imputer(strategy="median", inputCols=["price"], outputCols=["price_imputed"])
df_clean = price_imputer.fit(df_clean).transform(df_clean)
df_clean = df_clean.drop("price").withColumnRenamed("price_imputed", "price")

print("Null counts after preprocessing:")
df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["event_time_ts", "event_type", "product_id", "category_id",
              "category_code", "brand", "price", "user_id", "user_session"]
]).show()

print("Row count preserved:", df_clean.count())

Row count after preprocessing equals the original row count (42,448,764) — confirming **no rows
were dropped**, only values were filled or types converted.

---
## 9. Feature Engineering

### 9.1 Timestamp Features

In [ ]:
df_feat = (
    df_clean
    .withColumn("event_hour", F.hour("event_time_ts"))
    .withColumn("event_day", F.dayofmonth("event_time_ts"))
    .withColumn("event_dayofweek", F.dayofweek("event_time_ts"))  # 1=Sunday ... 7=Saturday
    .withColumn("event_month", F.month("event_time_ts"))
)

df_feat.select("event_time_ts", "event_hour", "event_day", "event_dayofweek", "event_month").show(5)

## 10. ML Dataset Level

The raw data is **one row per event** (view/cart/purchase). Individual events have no meaningful standalone
target — a single "view" row cannot be labeled "will purchase" or given a "spending" value on its own.
The natural unit of analysis for both the regression and classification tasks is the **session**
(`user_session`): a session aggregates a sequence of a user's events into a bounded browsing episode, which
is exactly the granularity at which "did this browsing episode end in a purchase?" and "how much did this
episode's purchases total?" are meaningful, non-degenerate questions. We therefore build a
**session-level feature table** and train both Spark MLlib models on it.

### 9.2 / 11 Session-Level Feature Engineering (Leakage-Safe)

In [ ]:
# IMPORTANT (data leakage prevention, see Section 14 for full explanation):
# All PREDICTOR features below are built ONLY from "view" and "cart" events -- i.e. signals that would
# realistically be available *before* we know whether/how much the session purchased. Purchase-derived
# quantities (purchase_count, total_spending) are computed separately and used ONLY as targets, never as
# input features.

pre_purchase = df_feat.filter(F.col("event_type").isin("view", "cart"))

session_features = (
    pre_purchase.groupBy("user_session")
    .agg(
        F.count("*").alias("session_event_count"),
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("session_view_count"),
        F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).alias("session_cart_count"),
        F.countDistinct("product_id").alias("session_unique_products"),
        F.countDistinct("brand").alias("session_unique_brands"),
        F.countDistinct("category_id").alias("session_unique_categories"),
        F.avg("price").alias("session_avg_price_viewed"),
        F.max("price").alias("session_max_price_viewed"),
        F.min("event_time_ts").alias("session_start"),
        F.max("event_time_ts").alias("session_end"),
        F.first("event_hour").alias("session_start_hour"),
        F.first("event_dayofweek").alias("session_dayofweek"),
    )
    .withColumn(
        "session_duration_seconds",
        F.col("session_end").cast("long") - F.col("session_start").cast("long")
    )
)

# Purchase-derived TARGET information, aggregated separately from the feature set above
purchase_agg = (
    df_feat.filter(F.col("event_type") == "purchase")
    .groupBy("user_session")
    .agg(
        F.count("*").alias("purchase_count"),
        F.sum("price").alias("session_total_spending")
    )
)

session_dataset = (
    session_features
    .join(purchase_agg, on="user_session", how="left")
    .fillna({"purchase_count": 0, "session_total_spending": 0.0})
    .withColumn("will_purchase", (F.col("purchase_count") > 0).cast("integer"))
)


# Cache session_dataset: it is used by regression, classification,
# SQL temp views, and multiple plot cells. Without caching each
# downstream action triggers a full recompute from HDFS.
session_dataset = session_dataset.cache()
print("Number of sessions:", session_dataset.count())  # materialises the cache
session_dataset.show(5, truncate=False)
session_dataset.show(5, truncate=False)

**Engineered features (all leakage-safe, derived only from `view`/`cart` events):**

- `session_event_count`, `session_view_count`, `session_cart_count` — activity volume within the session.
- `session_unique_products`, `session_unique_brands`, `session_unique_categories` — breadth of browsing.
- `session_avg_price_viewed`, `session_max_price_viewed` — price sensitivity / intent signal.
- `session_duration_seconds` — how long the browsing episode lasted.
- `session_start_hour`, `session_dayofweek` — temporal context of the session.

**Targets (computed separately, never fed back in as features):**

- `session_total_spending` — sum of `price` over `purchase` events in the session → regression target.
- `will_purchase` — 1 if the session contains at least one `purchase` event, else 0 → classification target.

## 14. Data Leakage Prevention

This is enforced structurally in Section 9.2/11 above, not just by convention:

- `session_features` (the feature table) is built **exclusively** from `view` and `cart` events
  (`pre_purchase = df_feat.filter(event_type in ('view','cart'))`).
- `purchase_agg` (the target table: `purchase_count`, `session_total_spending`) is built **exclusively**
  from `purchase` events, in a completely separate aggregation.
- The two are joined only at the very end, and `purchase_count` / `session_total_spending` are used **only**
  to define `will_purchase` and `session_total_spending` themselves — they are never included among the
  model's input feature columns.

Concretely: predicting `will_purchase` never uses `purchase_count` as an input (that would let the model
"see" the answer), and predicting `session_total_spending` never uses any purchase-event-derived quantity
as an input either. Every feature reflects only information that was observable from the user's browsing
(views/carts) — i.e. information that would realistically be available *before* the outcome is known.

---
## Save outputs for Stage 2 (ML) and Stage 3 (SQL)
_(Added for orchestration — not in the original notebook)_

In [ ]:
# --- Added for orchestration: persist cleaned data + session-level
# feature table so the ML and SQL notebooks can each load them
# independently (as separate Airflow tasks / separate Spark sessions).
df_clean.write.mode("overwrite").parquet(f"{DATA_DIR}/df_clean.parquet")
session_dataset.write.mode("overwrite").parquet(f"{DATA_DIR}/session_dataset.parquet")
print("Saved df_clean and session_dataset to", DATA_DIR)